# NOVA 02 — MOD-04 Currency Detection (MobileNetV3-Small)
**10 classes**: notes 500/1000/2000/5000/10000 + coins 25/50/100/200/500 (incl. the new BEAC Type-2024 coin series).

**Prerequisite:** upload `cfa_currency_kaggle.zip` (built by `scrape_cfa_images.py` + `augment_cfa_dataset.py` — official BEAC scans, augmented; 1830 train / 120 val / 120 test) as a **private Kaggle dataset**, attach it here, and set CFA_DATA below.

The bootstrap dataset uses clean official scans + augmentation. Coin classes have leaky val/test (few sources) — treat coin test metrics as optimistic until team-collected photos are added.

In [1]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

Cloning into '/kaggle/working/nova-ml'...


GPU OK: Tesla T4
Bootstrap OK — repo cloned, HF token loaded.


In [2]:
!pip install -q timm onnx2tf onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.

In [3]:
# Resolve the attached dataset mount (search 3 levels — Kaggle may nest
# under /kaggle/input/datasets/<owner>/<slug>)
import glob, os
inputs = (glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
          + glob.glob('/kaggle/input/*/*/*'))
CFA_DATA = next(p for p in inputs
                if 'cfa' in p.split('/')[-1].lower() and os.path.isdir(p))
# If the zip extracted with a wrapper folder, descend into it
if not os.path.isdir(f'{CFA_DATA}/train'):
    CFA_DATA = next(p for p in glob.glob(f'{CFA_DATA}/*')
                    if os.path.isdir(f'{p}/train'))
print('CFA_DATA =', CFA_DATA)
!ls {CFA_DATA}/train

CFA_DATA = /kaggle/input/datasets/fongebertin/cfa-currency-dataset
fcfa_coin_100  fcfa_coin_25  fcfa_coin_500   fcfa_note_10000  fcfa_note_500
fcfa_coin_200  fcfa_coin_50  fcfa_note_1000  fcfa_note_2000   fcfa_note_5000


In [4]:
# Two-phase: fine-tune EfficientNet-B4 teacher, distill into MobileNetV3-Small.
# Class count is auto-detected from the train/ folders (10).
!python scripts/train_currency_distillation.py --data-dir {CFA_DATA} \
    --teacher-epochs 20 --student-epochs 60 --batch-size 64

Classes (10): ['fcfa_coin_100', 'fcfa_coin_200', 'fcfa_coin_25', 'fcfa_coin_50', 'fcfa_coin_500', 'fcfa_note_1000', 'fcfa_note_10000', 'fcfa_note_2000', 'fcfa_note_500', 'fcfa_note_5000']
model.safetensors: 100%|███████████████████| 77.9M/77.9M [00:03<00:00, 22.7MB/s]
model.safetensors: 100%|███████████████████| 10.2M/10.2M [00:00<00:00, 16.7MB/s]
Phase 1: fine-tuning teacher (efficientnet_b4) for 20 epochs...
  teacher epoch 1/20
  teacher epoch 2/20
  teacher epoch 3/20
  teacher epoch 4/20
  teacher epoch 5/20
  teacher epoch 6/20
  teacher epoch 7/20
  teacher epoch 8/20
  teacher epoch 9/20
  teacher epoch 10/20
  teacher epoch 11/20
  teacher epoch 12/20
  teacher epoch 13/20
  teacher epoch 14/20
  teacher epoch 15/20
  teacher epoch 16/20
  teacher epoch 17/20
  teacher epoch 18/20
  teacher epoch 19/20
  teacher epoch 20/20
Teacher val accuracy: 0.9667
Phase 2: distilling into student (mobilenetv3_small_100) for 60 epochs...
  epoch 1/60 | loss 146.2337 | val_acc 0.7333
  epoc

In [5]:
# Held-out test evaluation at the 0.85 confidence gate (FR-04-03)
!python scripts/evaluate_models.py currency \
    --checkpoint /kaggle/working/checkpoints/currency_student_best.pth \
    --data-dir {CFA_DATA}/test

                 precision    recall  f1-score   support

  fcfa_coin_100       1.00      1.00      1.00         8
  fcfa_coin_200       1.00      1.00      1.00         8
   fcfa_coin_25       1.00      1.00      1.00         8
   fcfa_coin_50       1.00      1.00      1.00         8
  fcfa_coin_500       1.00      1.00      1.00         8
 fcfa_note_1000       1.00      1.00      1.00        16
fcfa_note_10000       1.00      1.00      1.00        16
 fcfa_note_2000       1.00      1.00      1.00        16
  fcfa_note_500       1.00      1.00      1.00        16
 fcfa_note_5000       1.00      1.00      1.00        16

       accuracy                           1.00       120
      macro avg       1.00      1.00      1.00       120
   weighted avg       1.00      1.00      1.00       120

{
  "overall_accuracy": 1.0,
  "accuracy_at_conf_0.85": 1.0,
  "coverage_at_conf_0.85": 0.9416666666666667
}


In [6]:
# Convert to TFLite INT8 (calibrate on val images) + benchmark
!python scripts/convert_to_tflite.py \
    --checkpoint /kaggle/working/checkpoints/currency_student_best.pth \
    --arch mobilenetv3_small_100 --num-classes 10 --input-size 224 \
    --out /kaggle/working/exports/currency_detection_v1.tflite \
    --calib-dir {CFA_DATA}/val --benchmark

W0711 03:13:32.956000 1251 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0711 03:13:33.711000 1251 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0711 03:13:33.712000 1251 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -

In [7]:
# Publish to HuggingFace: unixio/nova-currency-detection
!python scripts/push_to_huggingface.py --module MOD-04 \
    --pytorch /kaggle/working/checkpoints/currency_student_best.pth \
    --tflite /kaggle/working/exports/currency_detection_v1.tflite \
    --labels labels/cfa_labels.txt \
    --eval-json /kaggle/working/evaluation/currency_test_results.json --version 1.0.0

Traceback (most recent call last):
  File "/kaggle/working/nova-ml/scripts/push_to_huggingface.py", line 153, in <module>
    publish_model(args.module, args.pytorch, args.tflite, args.labels, eval_results, args.version)
  File "/kaggle/working/nova-ml/scripts/push_to_huggingface.py", line 100, in publish_model
    checksum = sha256_of(tflite_model)
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/nova-ml/scripts/nova_common.py", line 74, in sha256_of
    with open(file_path, "rb") as f:
         ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/exports/currency_detection_v1.tflite'
